In [15]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization Middleware

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [3]:
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model = "llama-3.3-70b-versatile",
    model_provider = "groq"
)                           
agent = create_agent(
    model = model,
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger = ("messages", 10),
            keep = ("messages", 4)
        )
    ]
    
)

In [17]:
config = {"configurable": {"thread_id": "test_1"}}

In [18]:
questions = [
    "what is 2+2",
    "what is 10/2",
    "what is 100/4",
    "who is dean jones",
    "what is 100+21*0+1"    
]

for i in questions:
    response = agent.invoke({"messages": [HumanMessage(content = i)]}, config)
    print(response["messages"][-1].content)

2+2 = 4
10/2 = 5
100/4 = 25
Dean Jones was an Australian cricketer. He played for the Australian national team from 1984 to 1994, and is best known for his impressive batting skills and his role in popularizing cricket in Australia. He passed away on September 24, 2020.
To calculate the expression 100+21**0+1, we need to follow the order of operations (PEMDAS):

1. **Multiply 21 and 0**: 21**0 = 0
2. **Add 100 and 0**: 100 + 0 = 100
3. **Add 1**: 100 + 1 = 101

So the final result is 101.


In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

In [20]:
@tool
def search_hotel(city: str) -> str:
    """Search hotels: returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, 350$ per night, spa, pool, gym.
    2. City Inn - 4 star , 180$ per night, business center.
    3. Budget Stay- 3 star, 75$ per night, free wifi"""

agent = create_agent(
    model = model,
    tools = [search_hotel],
    checkpointer= InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = model,
            trigger = ("tokens", 550),
            keep = ("tokens", 200)
        )
    ]

)

config = {"configurable" : {"thread_id": "test_2"}}

def token_count(messages):
    token_char = 0
    for message in messages:
        token_char += len(str(messages.content))
    return token_char // 4

In [ ]:
cities = ["Paris","London","Tokyo","New York", "Dubai", "Singapore"]
for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content= f"find hotels in {city}")]},
        config = config
    )

tokens = token_count(response["messages"])
print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
print(f"{(response['messages'])}")

In [14]:
for city in cities:
    print(f"Starting {city}...")

    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )

    print(f"Finished {city}")

Starting Paris...
Finished Paris
Starting London...


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_hotel={"city": "London"}</function>'}}